# SB100 Squad 2 — extração em lote

Extrai os PDFs **ainda não extraídos** e devolve o resultado ao Supabase.

Antes de começar:

1. **Ambiente de execução → Alterar tipo → GPU**
2. No ícone de chave (Secrets), crie três segredos com *Notebook access* ligado:
   `SUPABASE_URL`, `SUPABASE_ANON_KEY`, `SUPABASE_SERVICE_ROLE_KEY`

Se a sessão cair no meio, nada do que já subiu se perde: reabra e rode de novo,
que a seleção é por `extracted = false` e ele traz só o que falta.


## 1. Preparo


In [ ]:
from google.colab import userdata
import os, torch

print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available()
      else "SEM GPU - troque em Ambiente de execucao")

# sair da pasta ANTES de apagar: se a sessao anterior deixou o shell dentro
# dela, o rm puxa o tapete e todo comando seguinte falha com getcwd
%cd /content
!rm -rf /content/repo
!git clone -q https://github.com/nicolasaws1/parser_rag_framework.git /content/repo
%cd /content/repo
!pip install -q supabase python-dotenv pymupdf

# ------------------------------------------------------------------
# ABAIXO SAO OS NOMES DOS SEGREDOS, NAO AS CHAVES.
# Nao cole valor nenhum aqui. Os valores vao no icone de chave da barra
# lateral (Secrets), com Notebook access ligado em cada um.
# ------------------------------------------------------------------
NOMES = ("SUPABASE_URL", "SUPABASE_ANON_KEY", "SUPABASE_SERVICE_ROLE_KEY")

assert os.path.isdir("/content/repo"), "o clone falhou; rode esta celula de novo"
with open("/content/repo/.env", "w") as f:
    for nome in NOMES:
        valor = userdata.get(nome)
        assert valor, f"cadastre o segredo {nome} no icone de chave (Secrets)"
        print(f"{nome}={valor}", file=f)
print("ambiente pronto")


### Alternativa: subir o `.env` em vez de usar Secrets

Rode **esta** célula OU a de cima, não as duas. Aqui você escolhe o arquivo
`.env` da sua máquina (em `SB100\squad-2\.env`) e ele vai direto para o
Colab. Nenhuma chave é digitada, então não há como colar no lugar errado.

O arquivo fica só nesta sessão e some quando ela encerra.


In [ ]:
import os, shutil, torch
from google.colab import files

print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available()
      else "SEM GPU - troque em Ambiente de execucao")

%cd /content
!rm -rf /content/repo
!git clone -q https://github.com/nicolasaws1/parser_rag_framework.git /content/repo
%cd /content/repo
!pip install -q supabase python-dotenv pymupdf

print()
print("Escolha o arquivo .env da pasta SB100/squad-2:")
enviados = files.upload()
nome = next(iter(enviados))
shutil.move(f"/content/repo/{nome}", "/content/repo/.env")

linhas = [l for l in open("/content/repo/.env") if "=" in l and not l.startswith("#")]
tem = {l.split("=")[0].strip() for l in linhas}
for k in ("SUPABASE_URL", "SUPABASE_ANON_KEY", "SUPABASE_SERVICE_ROLE_KEY"):
    assert k in tem, f"o .env enviado nao tem {k}"
print(f"ambiente pronto: {len(linhas)} variaveis")


## 2. Ver o que falta

Só relata, não baixa nada.


In [ ]:
!python scripts/acervo.py 2>/dev/null | head -12
!python extractor/baixar_lote.py --quantos 0


## 3. Baixar o lote

Prioriza o que foi pedido pelo botão **Analisar** do site; depois os menores,
que dão retorno rápido e cabem numa sessão sem risco de perder tudo no meio.

Ele imprime a estimativa de GPU antes de você seguir.


In [ ]:
QUANTOS = 10   #@param {type:"integer"}
MENORES = True #@param {type:"boolean"}

extra = "--menores" if MENORES else ""
!python extractor/baixar_lote.py --quantos {QUANTOS} {extra}


## 4. Extrair

**SO_CHANDRA**: com `True`, o Docling sai por completo, não é nem instalado.
Hoje ele pega 21% das páginas (as com texto e sem tabela); as outras 79% já
vão no Chandra. Foi assim que o Boletim 100 rodou.

**CHANDRA_MAX_LADO**: o lado maior da imagem entregue ao modelo. Era 1800
fixo, o que encolhia uma A4 a 230 DPI e comia o detalhe fino, justamente
rótulo de eixo e expoente. 2200 usa a VRAM da L4 para ler melhor. O custo
cresce com a área: dobrar o lado quadruplica o tempo da página.

**LOTE_PAGINAS**: quantas páginas por chamada. `1` é o de sempre. Acima
disso a GPU processa em paralelo, o que ajuda se uma página sozinha não
satura a placa. **Não testei em GPU** — rode com 1 primeiro, depois 4 no
mesmo documento e compare o tempo impresso no fim. Se piorar ou quebrar,
volte para 1.

Se a sessão cair, rode de novo: pula o que já terminou.


In [ ]:
SO_CHANDRA = True        #@param {type:"boolean"}
CHANDRA_MAX_LADO = 2200  #@param {type:"integer"}
LOTE_PAGINAS = 1         #@param {type:"integer"}

import os
os.environ["SO_CHANDRA"] = "1" if SO_CHANDRA else "0"
os.environ["CHANDRA_MAX_LADO"] = str(CHANDRA_MAX_LADO)
os.environ["LOTE_PAGINAS"] = str(LOTE_PAGINAS)
!python extractor/extrator_colab.py


## 5. Mandar para o Supabase

`ingerir_extracao.py` **atualiza** o documento que já existe e não encosta em
`article_metadata`. Não troque pelo `ingest_supabase.py`: aquele apaga a linha
de `pdfs` e recria, e o cascade levaria junto os metadados vindos da curadoria,
que aqui não há de onde repor.

`gerar_figuras.py` recorta cada gráfico do PDF pela bbox do banco. É
idempotente, então só faz o que falta.


In [ ]:
!python scripts/ingerir_extracao.py /content/export
!python scripts/gerar_figuras.py --aplicar


## 6. Conferir

Depois disto, abra o site: os documentos do lote aparecem como extraídos, com
as figuras.


In [ ]:
!python scripts/acervo.py 2>/dev/null | head -12


## 7. Guardar uma cópia (opcional)

O resultado já está no Supabase. Isto é só para ter o arquivo bruto da corrida
na sua máquina.


In [ ]:
import shutil
shutil.make_archive('/content/extracao_lote', 'zip', '/content/export')
from google.colab import files
files.download('/content/extracao_lote.zip')
